# Merging schemas

This notebook writes two Parquet files with different schemas. The older file has `id` and `name`. The newer file adds `country`.

`mergeSchema` combines them into one result. Rows from the older file do not have `country`, so that column is null for those rows. The same merge is then applied when the newer file is appended to the `users_demo` Delta table.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA data_ingestion;

CREATE VOLUME IF NOT EXISTS raw_files;

In [0]:
from pyspark.sql import Row

base = "/Volumes/workspace/data_ingestion/raw_files/schema_demo"

# Old version: two columns
old_data = [
    Row(id=1, name="Alice"),
    Row(id=2, name="Bob")
]

# New version: three columns
new_data = [
    Row(id=3, name="Anna", country="Finland")
]

spark.createDataFrame(old_data).write.mode("overwrite") \
    .parquet(f"{base}/old/")

spark.createDataFrame(new_data).write.mode("overwrite") \
    .parquet(f"{base}/new/")

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/data_ingestion/raw_files/schema_demo/old/',
  format => 'parquet'
);

id,name,_rescued_data
1,Alice,null
2,Bob,null


In [0]:
base = "/Volumes/workspace/data_ingestion/raw_files/schema_demo"

df = (
    spark.read
    .option("mergeSchema", "true")
    .parquet(f"{base}/old/", f"{base}/new/")
)

df.printSchema()
display(df.orderBy("id"))

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- country: string (nullable = true)



id,name,country
1,Alice,null
2,Bob,null
3,Anna,Finland


In [0]:
%sql
CREATE OR REPLACE TABLE
  workspace.data_ingestion.users_demo (
    id BIGINT,
    name STRING
  )
USING DELTA;

In [0]:
%sql
INSERT INTO workspace.data_ingestion.users_demo
VALUES
  (1, 'Alice'),
  (2, 'Bob');

num_affected_rows,num_inserted_rows
2,2


In [0]:
new_df = spark.read.parquet(
    "/Volumes/workspace/data_ingestion/raw_files/schema_demo/new/"
)

(
    new_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("workspace.data_ingestion.users_demo")
)

In [0]:
%sql
SELECT *
FROM workspace.data_ingestion.users_demo
ORDER BY id;

id,name,country
1,Alice,null
2,Bob,null
3,Anna,Finland
